In [18]:
# =============================================================================
# SHOW REEL MEDIA GROUP — COMMUNITY PERSONA IDENTIFICATION PIPELINE
# Platform: Instagram Only
# Runtime: Google Colab Enterprise + Vertex AI (Gemini)
# Author: Show Reel Data Science Team
# Version: 1.0
#
# Pipeline Stages:
#   Stage 0 — Feature Engineering (user-level aggregation from ig_comments + ig_media)
#   Stage 1 — Exploratory Taxonomy Discovery (Gemini Flash, sample ~500 users)
#   Stage 2 — Deterministic CAG Classification (Gemini Pro, full dataset)
#
# Scheduling: Designed for Colab Enterprise Scheduled Runtime.
#   Set PIPELINE_MODE = "SAMPLE"  → Stage 0 + Stage 1 only (taxonomy discovery)
#   Set PIPELINE_MODE = "FULL"    → Stage 0 + Stage 2 only (production classification)
#   Set PIPELINE_MODE = "ALL"     → All stages in sequence
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────────────────
# Run this cell once per runtime, then restart.
!pip install -q google-cloud-aiplatform pandas numpy tqdm langdetect emoji

import subprocess, sys
# Uncomment below if running in a fresh Colab environment:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "google-cloud-aiplatform", "pandas", "numpy", "tqdm",
                 "langdetect", "emoji"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'google-cloud-aiplatform', 'pandas', 'numpy', 'tqdm', 'langdetect', 'emoji'], returncode=0)

In [19]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — CONFIGURATION (Toggle models and modes here)
# ─────────────────────────────────────────────────────────────────────────────

import os

# ── GCP / Vertex AI ──────────────────────────────────────────────────────────
GCP_PROJECT_ID   = "gen-lang-client-0792749758"          # TODO: set your project
GCP_LOCATION     = "us-central1"                   # Vertex AI region
GCS_BUCKET       = "afb_showreel"     # GCS path for outputs

# ── Model Toggle ─────────────────────────────────────────────────────────────
# Stage 1 (Exploratory): Flash is cheaper and fast for discovery on a sample
# Stage 2 (Deterministic): Pro for higher reasoning quality on full dataset
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"
# Swap the above to "gemini-2.0-flash-001" for both if cost is a constraint.

# ── Determinism Config (mandatory for academic reproducibility) ───────────────
TEMPERATURE = 0.0   # Greedy decoding — zero stochasticity
TOP_P       = 1.0   # No nucleus truncation; greedy is sole selector
MAX_OUTPUT_TOKENS_STAGE1 = 2048
MAX_OUTPUT_TOKENS_STAGE2 = 512   # Classification outputs are compact

# ── Pipeline Mode ────────────────────────────────────────────────────────────
# "SAMPLE" → Stage 0 + Stage 1 (taxonomy discovery, human-in-the-loop)
# "FULL"   → Stage 0 + Stage 2 (production classification, requires taxonomy JSON)
# "ALL"    → Both stages in sequence (not recommended for large datasets)
PIPELINE_MODE = "SAMPLE"   # Toggle here

# ── Data Paths ────────────────────────────────────────────────────────────────
# Adjust to your GCS or local paths
IG_COMMENTS_PATH = "ig_comments_cleaned.parquet"
IG_MEDIA_PATH    = "ig_media_cleaned.parquet"

# ── Sampling Config ───────────────────────────────────────────────────────────
SAMPLE_N_USERS        = 500    # Stage 1: number of users for taxonomy discovery
SAMPLE_SEED           = 42
BATCH_SIZE_STAGE1     = 20     # Users per exploratory LLM call (taxonomy)
BATCH_SIZE_STAGE2     = 10     # Users per classification call

# ── Artifact Paths ────────────────────────────────────────────────────────────
TAXONOMY_JSON_PATH    = "outputs/taxonomy.json"
CAG_CACHE_DIR         = "outputs/cag_cache/"       # Pickled CAG objects per post
RESULTS_PATH          = "outputs/user_personas.parquet"
os.makedirs("outputs", exist_ok=True)
os.makedirs(CAG_CACHE_DIR, exist_ok=True)

print("✅ Configuration loaded.")
print(f"   Mode: {PIPELINE_MODE}")
print(f"   Stage 1 model: {MODEL_STAGE1_EXPLORATORY}")
print(f"   Stage 2 model: {MODEL_STAGE2_CLASSIFY}")



✅ Configuration loaded.
   Mode: SAMPLE
   Stage 1 model: gemini-2.5-flash
   Stage 2 model: gemini-2.5-pro


In [20]:
# prompt: test whether the llms of the two stages would work on vertex ai

from google.cloud import aiplatform
from vertexai.generative_models import GenerativeModel, Part

# Initialize Vertex AI
aiplatform.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

# Load models
model_stage1 = GenerativeModel(MODEL_STAGE1_EXPLORATORY)
model_stage2 = GenerativeModel(MODEL_STAGE2_CLASSIFY)

# Define a simple prompt for testing
test_prompt = "What is the capital of France?"

print("Testing Stage 1 Model...")
try:
    response_stage1 = model_stage1.generate_content(
        test_prompt,
        generation_config={
            "max_output_tokens": MAX_OUTPUT_TOKENS_STAGE1,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
        }
    )
    print(f"Stage 1 Model Response: {response_stage1.text}")
    print("✅ Stage 1 model is working.")
except Exception as e:
    print(f"❌ Error testing Stage 1 model: {e}")

print("\nTesting Stage 2 Model...")
try:
    response_stage2 = model_stage2.generate_content(
        test_prompt,
        generation_config={
            "max_output_tokens": MAX_OUTPUT_TOKENS_STAGE2,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
        }
    )
    print(f"Stage 2 Model Response: {response_stage2.text}")
    print("✅ Stage 2 model is working.")
except Exception as e:
    print(f"❌ Error testing Stage 2 model: {e}")


Testing Stage 1 Model...


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Stage 1 Model Response: The capital of France is **Paris**.
✅ Stage 1 model is working.

Testing Stage 2 Model...
Stage 2 Model Response: The capital of France is **Paris**.
✅ Stage 2 model is working.


In [21]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — VERTEX AI INITIALIZATION
# ─────────────────────────────────────────────────────────────────────────────

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

def get_model(model_name: str) -> GenerativeModel:
    """Returns a configured GenerativeModel for the given model name."""
    return GenerativeModel(model_name)

def get_generation_config(max_tokens: int) -> GenerationConfig:
    """Returns a deterministic GenerationConfig."""
    return GenerationConfig(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_output_tokens=max_tokens,
        response_mime_type="application/json",  # Enforce JSON output at API level
    )

print("✅ Vertex AI initialized.")

✅ Vertex AI initialized.


In [36]:
import pandas as pd
# Define the directory and the files we found earlier
base_path = f"gs://{GCS_BUCKET}/"
files_to_read = [
    "ig_comments_cleanded.parquet",
]

datasets = {}

for file in files_to_read:
    path = base_path + file
    print(f"Reading {file}...")
    datasets[file] = pd.read_parquet(path)

print("\nAll datasets loaded into the 'datasets' dictionary.")
for name, data in datasets.items():
    print(f"{name}: {len(data)} rows")

ig_comments = datasets['ig_comments_cleaned.parquet']
ig_comments.head()
#

Reading ig_comments_with_llm.parquet...

All datasets loaded into the 'datasets' dictionary.
ig_comments_with_llm.parquet: 499752 rows


,comment_id,media_id,comment_text,like_count,timestamp,user_id,from_username,parent_id,is_reply,month,dayofweek,hour,is_creator,persona,confidence,trigger_phrase
0,18411796564134026,18093827120283415,Miticaaaaaa❤️❤️❤️❤️❤️❤️👏👏👏👏👏,3,2026-03-19 14:43:02,3482058005293604,giovannamirci,None,False,3,Thursday,14,False,Neutral,0.730407,question asked
1,18387262639084499,18093827120283415,"Divano comodo come una nuvola, dove puoi farci...",3,2026-03-19 13:39:14,3103734239813346,alessia.daziano,None,False,3,Thursday,13,False,Inquisitive,0.589957,call to action
2,18086945812969028,18093827120283415,Tu hai messo una sedia della Kartell in ultima...,32,2026-03-19 13:39:00,908266745384508,guglielmoscilla,None,False,3,Thursday,13,False,Neutral,0.141560,strong sentiment
3,18075175775440194,18093827120283415,@guglielmoscilla E QUANTO NE VADO FIERA NON HA...,10,2026-03-19 13:55:56,17841401147950242,camihawke,18086945812969028,True,3,Thursday,13,True,Positive,0.066723,product mention
4,18114582079661452,18093827120283415,"@guglielmoscilla Gu, io da gaymer concordo con...",1,2026-03-19 21:29:53,1709620023622902,alex_danci,18086945812969028,True,3,Thursday,21,False,Opinionated,0.191945,product mention


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — DATA LOADING & VALIDATION
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import os

# Define the directory and the files we found earlier
base_path = f"gs://{GCS_BUCKET}/"

print(f"ፃ Accessing bucket: {base_path}")

# Loading Comments
print(f"Reading ig_comments_cleaned.parquet...")
ig_comments = pd.read_parquet(base_path + "ig_comments_cleaned.parquet")

# Loading Posts/Media
print(f"Reading ig_posts_cleaned.parquet as ig_media...")
ig_media = pd.read_parquet(base_path + "ig_posts_cleaned.parquet")

print(f"\n✅ Datasets loaded.")
print(f"   ig_comments: {len(ig_comments):,} rows")
print(f"   ig_media:    {len(ig_media):,} rows")

# ── Validation ────────────────────────────────────────────────────────────
# Filter comments
n_before_c = len(ig_comments)
ig_comments = ig_comments[
    (ig_comments["text"].notna()) &
    (ig_comments["text"].str.strip() != "") &
    (ig_comments["from_id"].notna())
].copy()

# Filter media (ensure media_id is present for merging)
ig_media = ig_media[ig_media["media_id"].notna()].copy()

print(f"\nፁ Post-Validation Summary:")
print(f"   Comments retained: {len(ig_comments):,} / {n_before_c:,}")
print(f"   Unique commenters: {ig_comments['from_id'].nunique():,}")
print(f"   Unique media IDs:  {ig_media['media_id'].nunique():,}")

print("\n Comment Dataset")
display(ig_comments.head(2))
print("`\n Post Dataset")
display(ig_media.head(2))

ፃ Accessing bucket: gs://afb_showreel/
Reading ig_comments_cleaned.parquet...
Reading ig_posts_cleaned.parquet as ig_media...

✅ Datasets loaded.
   ig_comments: 499,752 rows
   ig_media:    1,540 rows

ፁ Post-Validation Summary:
   Comments retained: 498,576 / 499,752
   Unique commenters: 194,512
   Unique media IDs:  1,540

 Comment Dataset


,comment_id,media_id,text,like_count,timestamp,from_id,from_username,parent_id,is_reply,month,dayofweek,hour,is_creator
0,18411796564134026,18093827120283415,Miticaaaaaa❤️❤️❤️❤️❤️❤️👏👏👏👏👏,3,2026-03-19 14:43:02,3482058005293604,giovannamirci,None,False,3,Thursday,14,False
1,18387262639084499,18093827120283415,"Divano comodo come una nuvola, dove puoi farci...",3,2026-03-19 13:39:14,3103734239813346,alessia.daziano,None,False,3,Thursday,13,False


`
 Post Dataset


,media_id,caption,comments_count,like_count,media_product_type,media_type,permalink,timestamp,reach,saved,views,total_interactions,comments_downloaded,year,Not_reel,month,dayofweek,hour
0,17941939319341563,Pronta a fissarmi goal irraggiungibili anche q...,125,43223,REELS,VIDEO,https://www.instagram.com/reel/Cm6WPZRhfBB/,2023-01-02 11:53:19,906443.0,1212.0,1138605.0,44602.0,1,2023,False,1,Monday,11
1,17992425124653410,2023: ritagliarsi ancora piu tempo (prezioso) ...,34,43776,FEED,CAROUSEL_ALBUM,https://www.instagram.com/p/Cm9ZVRxj1jO/,2023-01-03 00:00:00,373101.0,160.0,0.0,43970.0,-1,2023,True,1,Tuesday,0


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.5 — EARLY STAGE 0: COMMENT TEXT PROCESSING & JSONL EXPORT
# ─────────────────────────────────────────────────────────────────────────────
# Process and export raw comments to JSONL immediately after data loading.
# This ensures processed comments are available in GCS BEFORE Stage 1/2 runs.

import json
import re
from typing import Optional, Dict, Any

def advanced_text_preprocessing(text: str) -> Dict[str, Any]:
    """
    Comprehensive comment text preprocessing:
    - Normalizes whitespace
    - Removes URLs
    - Preserves emoji but extracts count
    - Detects language
    - Flags mentions and hashtags
    """
    if not isinstance(text, str) or not text.strip():
        return {
            "text_original": "",
            "text_cleaned": "",
            "text_no_emoji": "",
            "language": None,
            "emoji_count": 0,
            "mention_count": 0,
            "hashtag_count": 0,
            "is_reply_text": False,
            "has_multiple_languages": False,
        }

    text_orig = text.strip()

    # Remove URLs (http/https/www patterns)
    text_no_urls = re.sub(r'https?://\S+|www\.\S+', '', text_orig)

    # Count and extract emoji
    emoji_count = emoji_lib.emoji_count(text_no_urls)

    # Remove emoji for strict text processing
    text_no_emoji_version = emoji_lib.replace_emoji(text_no_urls, "")

    # Normalize whitespace
    text_cleaned = re.sub(r'\s+', ' ', text_no_urls).strip()
    text_no_emoji = re.sub(r'\s+', ' ', text_no_emoji_version).strip()

    # Count mentions and hashtags
    mention_count = len(re.findall(r'@\w+', text_cleaned))
    hashtag_count = len(re.findall(r'#\w+', text_cleaned))

    # Check if text starts with mention
    is_reply_text = bool(re.match(r'^\s*@\w+', text_orig))

    # Language detection
    try:
        detected_lang = langdetect.detect(text_no_emoji if text_no_emoji.strip() else "x")
    except:
        detected_lang = None

    # Detect if text contains multiple script blocks
    has_multiple_scripts = bool(
        re.search(r'[a-zA-Z]', text_cleaned) and
        re.search(r'[^\x00-\x7F]', text_cleaned)
    )

    return {
        "text_original": text_orig,
        "text_cleaned": text_cleaned,
        "text_no_emoji": text_no_emoji,
        "language": detected_lang,
        "emoji_count": emoji_count,
        "mention_count": mention_count,
        "hashtag_count": hashtag_count,
        "is_reply_text": is_reply_text,
        "has_multiple_languages": has_multiple_scripts,
    }


def export_comments_to_jsonl(
    comments_df: pd.DataFrame,
    output_gcs_path: str,
    max_records: Optional[int] = None,
) -> int:
    """Export processed comments to JSONL format for LLM consumption."""
    from google.cloud import storage

    print(f"\n📝 Processing {len(comments_df):,} comments for JSONL export...")

    df = comments_df.copy()
    if max_records:
        df = df.sample(n=min(max_records, len(df)), random_state=42)

    # Apply advanced preprocessing
    print("   ⚙️  Running advanced text preprocessing...")
    text_features = df["text"].apply(advanced_text_preprocessing)

    # Create JSONL records
    records = []
    for idx, (_, row) in enumerate(df.iterrows()):
        text_proc = text_features.iloc[idx]

        metadata = {
            "language": text_proc["language"],
            "emoji_count": int(text_proc["emoji_count"]),
            "mention_count": int(text_proc["mention_count"]),
            "hashtag_count": int(text_proc["hashtag_count"]),
            "is_reply": bool(text_proc["is_reply_text"]),
            "timestamp": str(row.get("timestamp", "")),
            "likes": int(row.get("like_count", 0)),
            "media_id": str(row.get("media_id", "")),
        }

        record = {
            "id": str(row.get("comment_id", f"comment_{idx}")),
            "user_id": str(row.get("from_id", "")),
            "username": str(row.get("from_username", "")),
            "text_original": text_proc["text_original"],
            "text_cleaned": text_proc["text_cleaned"],
            "text_no_emoji": text_proc["text_no_emoji"],
            "metadata": metadata,
        }
        records.append(record)

    # Write to JSONL
    jsonl_buffer = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)

    # Upload to GCS
    print(f"   ☁️  Uploading to {output_gcs_path}...")
    try:
        client = storage.Client()
        bucket_name = output_gcs_path.replace("gs://", "").split("/")[0]
        blob_path = "/".join(output_gcs_path.replace("gs://", "").split("/")[1:])

        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_path)
        blob.upload_from_string(jsonl_buffer, content_type="application/x-ndjson")

        print(f"   ✅ Exported {len(records):,} records to: {output_gcs_path}")
        return len(records)
    except Exception as e:
        print(f"   ⚠️  GCS upload failed: {e}")
        print(f"   💾 Falling back to local export...")
        local_path = output_gcs_path.replace("gs://", "").replace("/", "_")
        with open(local_path, "w", encoding="utf-8") as f:
            f.write(jsonl_buffer)
        print(f"   ✅ Exported {len(records):,} records to: {local_path}")
        return len(records)


# ── EXECUTE: Export processed comments to JSONL ────────────────────────────────
print("⏳ Exporting processed comments to JSONL (GCS)...")
export_comments_to_jsonl(
    comments_df=ig_comments,
    output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm.jsonl",
    max_records=None
)

print("\n✅ JSONL export complete. Processed comments ready for Stage 1/2.")


In [38]:
!gsutil ls gs://{GCS_BUCKET}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://afb_showreel/YTcomments_1_cleaned.parquet
gs://afb_showreel/YTcomments_2_cleaned.parquet
gs://afb_showreel/YTcomments_3_cleaned.parquet
gs://afb_showreel/YTcomments_4_cleaned.parquet
gs://afb_showreel/channels_metadata.csv
gs://afb_showreel/fb_comments_clean.parquet
gs://afb_showreel/fb_posts_clean.parquet
gs://afb_showreel/ig_comments_cleaned.parquet
gs://afb_showreel/ig_posts_cleaned.parquet
gs://afb_showreel/processed_urls.txt
gs://afb_showreel/tk_comments_clean.parquet
gs://afb_showreel/tk_posts_clean.parquet
gs://afb_showreel/undownloaded_links.csv
gs://afb_showreel/videos_metadata.csv
gs://afb_showreel/2519586145507999744/
gs://afb_showreel/3488985965299499008/
gs://afb_showreel/Output/
gs://afb_showreel/cluster_summarie

In [39]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — STAGE 0: FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
# Constructs a user-level feature matrix from raw comment and media data.

import re
import emoji as emoji_lib
from datetime import timedelta

# ── 4.1 Comment-level features ───────────────────────────────────────

def compute_comment_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["char_length"]  = df["text"].str.len()
    df["word_count"]   = df["text"].str.split().str.len()
    df["emoji_count"]  = df["text"].apply(lambda t: emoji_lib.emoji_count(str(t)))
    df["has_emoji"]    = (df["emoji_count"] > 0).astype(int)
    df["has_question"] = df["text"].str.contains(r"\?", na=False).astype(int)
    df["has_exclaim"]  = df["text"].str.contains(r"!", na=False).astype(int)
    df["mention_count"] = df["text"].str.count(r"@\w+")
    df["hashtag_count"] = df["text"].str.count(r"#\w+")
    df["is_reply"] = df["parent_id"].notna().astype(int)
    return df

ig_comments = compute_comment_features(ig_comments)
print("✅ Comment-level features computed.")

# ── 4.2 Merge with media for temporal features ─────────────────────────────

# Dynamically select columns that exist in ig_media
media_cols = ["media_id", "timestamp", "like_count", "comments_count", "media_product_type", "media_type", "engagement_rate"]
existing_media_cols = [c for c in media_cols if c in ig_media.columns]

ig_comments = ig_comments.merge(
    ig_media[existing_media_cols].rename(
        columns={"timestamp": "post_timestamp",
                 "like_count": "post_like_count",
                 "comments_count": "post_comments_count"}
    ),
    on="media_id", how="left"
)

# Ensure timestamp conversion
ig_comments["timestamp"] = pd.to_datetime(ig_comments["timestamp"])
ig_comments["post_timestamp"] = pd.to_datetime(ig_comments["post_timestamp"])

ig_comments["hours_to_comment"] = (
    (ig_comments["timestamp"] - ig_comments["post_timestamp"])
    .dt.total_seconds() / 3600
).clip(lower=0)

# ── 4.3 User-level aggregation ───────────────────────────────────

def build_user_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby("from_id")

    # Handle missing engagement_rate column for aggregation
    agg_dict = {
        "from_username":           grp["from_username"].last(),
        "total_comments":          grp.size(),
        "unique_posts_commented":  grp["media_id"].nunique(),
        "total_replies_made":      grp["is_reply"].sum(),
        "reply_ratio":             grp["is_reply"].mean(),
        "mean_hours_to_comment":   grp["hours_to_comment"].mean(),
        "median_hours_to_comment": grp["hours_to_comment"].median(),
        "pct_comments_under_1h":   grp["hours_to_comment"].apply(lambda x: (x < 1).mean()),
        "pct_comments_under_24h":  grp["hours_to_comment"].apply(lambda x: (x < 24).mean()),
        "activity_span_days":      grp["timestamp"].apply(lambda x: (x.max() - x.min()).days),
        "total_likes_received":    grp["like_count"].sum(),
        "mean_likes_per_comment":  grp["like_count"].mean(),
        "mean_word_count":         grp["word_count"].mean(),
        "emoji_usage_rate":        grp["has_emoji"].mean(),
        "question_rate":           grp["has_question"].mean(),
        "exclamation_rate":        grp["has_exclaim"].mean()
    }

    if "engagement_rate" in df.columns:
        agg_dict["mean_post_engagement_rate"] = grp["engagement_rate"].mean()

    feat = pd.DataFrame(agg_dict).reset_index()
    feat["post_concentration_ratio"] = (feat["unique_posts_commented"] / feat["total_comments"]).clip(upper=1.0)
    return feat

user_features = build_user_feature_matrix(ig_comments)
print(f"✅ User feature matrix built: {len(user_features):,} users.")

# ── 4.4 Top comments for LLM ───────────────────────────────────

top_comments_per_user = (
    ig_comments.sort_values("like_count", ascending=False)
    .groupby("from_id").head(5)
    .groupby("from_id")["text"].apply(lambda texts: " ||| ".join(texts.astype(str).tolist()))
    .reset_index().rename(columns={"text": "top_comments_sample"})
)

user_features = user_features.merge(top_comments_per_user, on="from_id", how="left")
print("✅ Representative comment history attached.")

✅ Comment-level features computed.
✅ User feature matrix built: 194,512 users.
✅ Representative comment history attached.


In [40]:
from vertexai.preview import caching
import datetime

# ────────────────────────────────
# CELL 5.1 — OPTIMIZED CAG (EXPLICIT VERTEX AI CONTEXT CACHING)
# ────────────────────────────────

def create_explicit_vertex_cache(system_instruction: str, context_text: str, model_name: str = MODEL_STAGE2_CLASSIFY):
    """
    Creates a server-side KV cache for the provided context and system instructions.
    This avoids re-tokenizing the context for every request in a batch.
    """
    print(f"⚙ Pre-computing Vertex AI Context Cache (TTL: 1h)...")

    cached_content = caching.CachedContent.create(
        model_name=model_name,
        system_instruction=system_instruction,
        contents=[context_text],
        ttl=datetime.timedelta(hours=1)
    )
    print(f"✅ Cache created: {cached_content.name}")
    return cached_content

print("✅ Vertex AI Explicit Caching logic (Optimized) ready.")

✅ Vertex AI Explicit Caching logic (Optimized) ready.


In [45]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — STAGE 1: EXPLORATORY TAXONOMY DISCOVERY (SAMPLE)
# ─────────────────────────────────────────────────────────────────────────────
# Based on the 3-step pipeline from Boughanmi et al. (2025) as referenced
# in the Show Reel LLMs & Prompting brief.
#
# Process:
#   1. Sample N users from the full user feature matrix
#   2. Send batches of user profiles to Gemini Flash
#   3. LLM identifies latent behavioral patterns / archetypes
#   4. Human-in-the-loop consolidation into a MECE taxonomy
#   5. Taxonomy saved to taxonomy.json for Stage 2
# ─────────────────────────────────────────────────────────────────────────────



In [41]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — STAGE 1: EXPLORATORY TAXONOMY DISCOVERY (SAMPLE)
# ─────────────────────────────────────────────────────────────────────────────
# Based on the 3-step pipeline from Boughanmi et al. (2025) as referenced
# in the Show Reel LLMs & Prompting brief.
#
# Process:
#   1. Sample N users from the full user feature matrix
#   2. Send batches of user profiles to Gemini Flash
#   3. LLM identifies latent behavioral patterns / archetypes
#   4. Human-in-the-loop consolidation into a MECE taxonomy
#   5. Taxonomy saved to taxonomy.json for Stage 2
# ─────────────────────────────────────────────────────────────────────────────

import json
import time
from tqdm import tqdm

STAGE1_SYSTEM_PROMPT = """You are an expert community analyst for a major Italian influencer agency.
Your task is to identify distinct audience persona archetypes from user comment behaviour data.

You will receive a batch of Instagram commenter profiles. Each profile contains:
- Quantitative behavioral metrics (comment frequency, timing, engagement)
- A sample of their actual comments (pipe-separated)

Your goal is to identify RECURRING behavioral patterns across these users.
For each pattern you observe, output a candidate persona with:
- A short codename (e.g., "SUPERFAN", "BRAND_ADVOCATE")
- A 1-sentence behavioral description
- Key distinguishing quantitative signals
- 2-3 representative verbatim comment fragments

Output ONLY a valid JSON array of persona objects. No preamble, no markdown fences.
Schema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]"""


def format_user_profile_for_stage1(row: pd.Series) -> str:
    """Format a user feature row as a compact text profile for the LLM."""
    return (
        f"USER: {row['from_username']} | "
        f"Total comments: {int(row['total_comments'])} | "
        f"Unique posts: {int(row['unique_posts_commented'])} | "
        f"Activity span: {int(row['activity_span_days'])} days | "
        f"Avg hrs to comment: {row['mean_hours_to_comment']:.1f}h | "
        f"Early commenter (<1h): {row['pct_comments_under_1h']:.0%} | "
        f"Reply ratio: {row['reply_ratio']:.0%} | "
        f"Avg likes/comment: {row['mean_likes_per_comment']:.1f} | "
        f"Avg word count: {row['mean_word_count']:.0f} | "
        f"Emoji rate: {row['emoji_usage_rate']:.0%} | "
        f"Question rate: {row['question_rate']:.0%} | "
        f"Sample comments: {str(row.get('top_comments_sample', ''))[:400]}"
    )


def run_stage1_taxonomy_discovery(
    user_features_df: pd.DataFrame,
    n_sample: int = SAMPLE_N_USERS,
    batch_size: int = BATCH_SIZE_STAGE1,
    seed: int = SAMPLE_SEED,
) -> list[dict]:
    """
    Stage 1: Feed user profile batches to Gemini Flash and collect
    candidate persona descriptions. Returns raw list of all candidates
    for human-in-the-loop consolidation.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 1 — Exploratory Taxonomy Discovery")
    print(f"  Sample: {n_sample} users | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE1_EXPLORATORY}")
    print(f"{'='*60}\n")

    sample_df = user_features_df.sample(
        n=min(n_sample, len(user_features_df)),
        random_state=seed
    ).reset_index(drop=True)

    model   = get_model(MODEL_STAGE1_EXPLORATORY)
    config  = get_generation_config(MAX_OUTPUT_TOKENS_STAGE1)

    all_candidates = []
    batches = [sample_df.iloc[i:i+batch_size]
               for i in range(0, len(sample_df), batch_size)]

    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 1 batches")):
        profiles_text = "\n\n".join(
            format_user_profile_for_stage1(row)
            for _, row in batch.iterrows()
        )

        prompt = f"""{STAGE1_SYSTEM_PROMPT}

--- USER PROFILES BATCH {batch_idx + 1} of {len(batches)} ---

{profiles_text}

Identify all distinct behavioral archetypes present in this batch.
Output ONLY the JSON array."""

        try:
            response = model.generate_content(
                contents=prompt,
                generation_config=config,
            )
            raw = response.text.strip()
            # Defensive JSON parsing
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            candidates = json.loads(raw)
            if isinstance(candidates, list):
                all_candidates.extend(candidates)
            print(f"   Batch {batch_idx+1}: {len(candidates)} candidates found.")
        except Exception as e:
            print(f"   ⚠️  Batch {batch_idx+1} error: {e}")

        time.sleep(1.5)  # Rate limit buffer

    print(f"\n✅ Stage 1 complete. Total raw candidates: {len(all_candidates)}")
    return all_candidates


def save_taxonomy_for_review(candidates: list[dict],
                              output_path: str = TAXONOMY_JSON_PATH):
    """
    Save raw candidates to JSON for human-in-the-loop consolidation.
    The researcher reviews this file, merges/prunes into a MECE taxonomy,
    and overwrites the file before running Stage 2.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    output = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": (
            "Review and consolidate the candidates below into a MECE taxonomy. "
            "Each final persona must have a unique 'codename'. "
            "Change status to 'APPROVED' before running Stage 2."
        ),
        "raw_candidates": candidates,
        "final_taxonomy": []  # Human fills this in
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"✅ Raw taxonomy candidates saved to: {output_path}")
    print("   ⚠️  HUMAN ACTION REQUIRED: Review, consolidate, and set status='APPROVED'.")


# ── EXAMPLE: What the final_taxonomy section should look like after review ───
# (This serves as the few-shot grounding for Stage 2)
EXAMPLE_MECE_TAXONOMY = [
    {
        "codename": "HYPER_ENGAGED_SUPERFAN",
        "label": "Hyper-Engaged Superfan",
        "description": "Comments within the first hour on nearly every post. Uses rich, emotional language, high emoji density, and frequently references the creator by name or inside community jokes. The emotional anchor of the community.",
        "quantitative_signals": ["pct_comments_under_1h > 0.60", "total_comments > 30", "activity_span_days > 180"],
        "example_comments": ["Sei la mia persona preferita ❤️🔥", "Non vedo l'ora ogni volta che posti!", "Da quando ti seguo non riesco a smettere 😭"]
    },
    {
        "codename": "CONSTRUCTIVE_CRITIC",
        "label": "Constructive Critic",
        "description": "Long, thoughtful comments with questions. Engages selectively but substantively. Often challenges the content or creator's viewpoints respectfully. High word count, high question rate.",
        "quantitative_signals": ["mean_word_count > 25", "question_rate > 0.40", "total_comments between 5 and 30"],
        "example_comments": ["Non sono d'accordo sul punto X, secondo me...", "Hai considerato anche la prospettiva...?"]
    },
    {
        "codename": "BRAND_CHAMPION",
        "label": "Brand Champion",
        "description": "Concentrates activity on sponsored/AD posts. Mentions product names, uses transactional language, tags friends. Likely conversion-oriented.",
        "quantitative_signals": ["pct_on_reels > 0.70", "mention_count > avg", "comments cluster on AD posts"],
        "example_comments": ["Ho già comprato! Fantastico 🛒", "Ti taggo @amico, devi vederlo!", "Codice sconto funziona?"]
    },
    {
        "codename": "PASSIVE_CONSUMPTIVE_OBSERVER",
        "label": "Passive Consumptive Observer",
        "description": "Comments rarely, very short (1-3 words, single emoji). Low emotional investment but consistently present across a long timespan. Silent majority.",
        "quantitative_signals": ["mean_word_count < 5", "total_comments < 10", "activity_span_days > 90"],
        "example_comments": ["❤️", "😂😂", "bellissimo"]
    },
    {
        "codename": "COMMUNITY_CONNECTOR",
        "label": "Community Connector",
        "description": "High reply ratio and mention count. Facilitates dialogue between other users. Acts as a social bridge within the comments section.",
        "quantitative_signals": ["reply_ratio > 0.50", "mean_mention_count > 1.5"],
        "example_comments": ["@utente1 hai visto questo?", "Come diceva @utente2 ieri...", "Rispondo a tutti! Chi ha domande?"]
    },
]

print("✅ Example MECE taxonomy defined (used as few-shot context in Stage 2).")



✅ Example MECE taxonomy defined (used as few-shot context in Stage 2).


In [27]:
# prompt: show the dataset that has the output of the llm from cell above

import pandas as pd
if PIPELINE_MODE in ["SAMPLE", "ALL"]:
    # Check if the taxonomy file exists and is approved
    if os.path.exists(TAXONOMY_JSON_PATH):
        with open(TAXONOMY_JSON_PATH, "r", encoding="utf-8") as f:
            taxonomy_data = json.load(f)
            if taxonomy_data.get("status") == "APPROVED" and taxonomy_data.get("final_taxonomy"):
                print("✅ Taxonomy is approved and loaded.")
                # The final_taxonomy is what we need to display
                final_taxonomy_df = pd.DataFrame(taxonomy_data["final_taxonomy"])
                print("\nDisplaying the approved final taxonomy:")
                display(final_taxonomy_df)
            elif taxonomy_data.get("status") == "PENDING_HUMAN_REVIEW":
                print("⚠️ Taxonomy is pending human review. Please review and approve the taxonomy.json file.")
                print("   Displaying raw candidates for your convenience:")
                raw_candidates_df = pd.DataFrame(taxonomy_data.get("raw_candidates", []))
                display(raw_candidates_df)
            else:
                print("⚠️ Taxonomy file exists but is not approved or is empty. Running Stage 1 to generate candidates.")
                # If not approved, we might want to re-run Stage 1 or prompt the user to review
                # For this task, we'll assume if it's not approved, we show raw candidates if available
                raw_candidates_df = pd.DataFrame(taxonomy_data.get("raw_candidates", []))
                if not raw_candidates_df.empty:
                    print("   Displaying raw candidates from previous Stage 1 run:")
                    display(raw_candidates_df)
                else:
                    print("   No raw candidates found. Please run Stage 1 first.")
    else:
        print("⚠️ Taxonomy file not found. Please run Stage 1 to generate taxonomy candidates.")
else:
    print("Pipeline mode is not set to 'SAMPLE' or 'ALL'. Stage 1 (taxonomy discovery) is skipped.")


⚠️ Taxonomy file not found. Please run Stage 1 to generate taxonomy candidates.


In [42]:


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — STAGE 2: DETERMINISTIC CAG CLASSIFICATION
# ─────────────────────────────────────────────────────────────────────────────
# Applies the approved taxonomy to classify ALL users in the dataset.
# Each user gets: persona_codename, confidence (0–1), justification_snippet.
#
# CAG Architecture:
#   - The taxonomy is compiled into a static system prompt (the "cache layer")
#   - User profiles are sent in batches as the variable "user turn"
#   - For post-level context, the PostCAGContext pickle is loaded per post
#     and prepended to the batch, anchoring semantic grounding
# ─────────────────────────────────────────────────────────────────────────────

def build_stage2_system_prompt(taxonomy: list[dict]) -> str:
    """
    Compile the MECE taxonomy into the static CAG system prompt.
    This is the immutable 'context foundation layer'.
    """
    taxonomy_text = ""
    for persona in taxonomy:
        taxonomy_text += f"""
PERSONA: {persona['codename']} — {persona['label']}
  Description: {persona['description']}
  Quantitative signals: {'; '.join(persona.get('quantitative_signals', []))}
  Example comments: {' | '.join(persona.get('example_comments', []))}
"""

    return f"""You are a deterministic community analyst for Show Reel Media Group.
Your task is to classify Instagram commenters into exactly ONE persona from the approved taxonomy.

=== APPROVED PERSONA TAXONOMY ===
{taxonomy_text}
=================================

CLASSIFICATION RULES:
1. Each user MUST be assigned to exactly ONE persona — the closest match.
2. You MUST output a confidence score between 0.0 and 1.0.
3. You MUST cite a specific text fragment from the user's comments as justification.
4. If a user's data is insufficient, assign the most probable persona with confidence ≤ 0.4.
5. Output ONLY a valid JSON array. No preamble, no markdown fences.

JSON Output Schema per user:
{{
  "from_id": str,
  "persona_codename": str,       // must be one of the taxonomy codenames
  "confidence": float,           // 0.0 to 1.0
  "justification": str           // 1 sentence citing evidence from comments/metrics
}}"""


def format_user_profile_for_stage2(row: pd.Series) -> dict:
    """Format user profile as a structured dict for Stage 2 classification."""
    return {
        "from_id": str(row["from_id"]),
        "from_username": str(row["from_username"]),
        "total_comments": int(row["total_comments"]),
        "unique_posts": int(row["unique_posts_commented"]),
        "activity_span_days": int(row["activity_span_days"]),
        "pct_comments_under_1h": round(float(row["pct_comments_under_1h"]), 2),
        "pct_comments_under_24h": round(float(row["pct_comments_under_24h"]), 2),
        "reply_ratio": round(float(row["reply_ratio"]), 2),
        "mean_likes_per_comment": round(float(row["mean_likes_per_comment"]), 2),
        "mean_word_count": round(float(row["mean_word_count"]), 1),
        "emoji_usage_rate": round(float(row["emoji_usage_rate"]), 2),
        "question_rate": round(float(row["question_rate"]), 2),
        "exclamation_rate": round(float(row["exclamation_rate"]), 2),
        "mean_mention_count": round(float(row["mean_mention_count"]), 2),
        "post_concentration_ratio": round(float(row["post_concentration_ratio"]), 2),
        "sample_comments": str(row.get("top_comments_sample", ""))[:500],
    }


def run_stage2_classification(
    user_features_df: pd.DataFrame,
    taxonomy: list[dict],
    ig_media_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE_STAGE2,
    output_path: str = RESULTS_PATH,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Stage 2: Classify all users deterministically using the approved taxonomy.
    Supports resumption from checkpoint to handle scheduled runtime interruptions.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 2 — Deterministic CAG Classification")
    print(f"  Total users: {len(user_features_df):,} | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE2_CLASSIFY} | Temp: {TEMPERATURE}")
    print(f"{'='*60}\n")

    # ── Resume from checkpoint ────────────────────────────────────────────────
    checkpoint_path = output_path.replace(".parquet", "_checkpoint.parquet")
    if resume and os.path.exists(checkpoint_path):
        done_df    = pd.read_parquet(checkpoint_path)
        done_ids   = set(done_df["from_id"].astype(str).tolist())
        todo_df    = user_features_df[~user_features_df["from_id"].isin(done_ids)]
        results    = done_df.to_dict("records")
        print(f"   ↺  Resuming from checkpoint: {len(done_ids):,} already classified, "
              f"{len(todo_df):,} remaining.")
    else:
        todo_df = user_features_df.copy()
        results = []

    # ── Build static CAG system prompt (compiled once) ────────────────────────
    system_prompt = build_stage2_system_prompt(taxonomy)

    model  = get_model(MODEL_STAGE2_CLASSIFY)
    config = get_generation_config(MAX_OUTPUT_TOKENS_STAGE2)

    batches     = [todo_df.iloc[i:i+batch_size]
                   for i in range(0, len(todo_df), batch_size)]
    error_count = 0

    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 2 classification")):
        user_profiles = [format_user_profile_for_stage2(row)
                         for _, row in batch.iterrows()]

        # ── Compose prompt: static CAG context + variable user batch ──────────
        user_batch_text = json.dumps(user_profiles, ensure_ascii=False, indent=2)

        full_prompt = f"""{system_prompt}

=== USER PROFILES TO CLASSIFY (Batch {batch_idx + 1} of {len(batches)}) ===
{user_batch_text}

Classify each user into exactly one persona. Output the JSON array only."""

        try:
            response = model.generate_content(
                contents=full_prompt,
                generation_config=config,
            )
            raw = response.text.strip()
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            classified = json.loads(raw)

            if isinstance(classified, list):
                results.extend(classified)

            # ── Save checkpoint every 10 batches ──────────────────────────────
            if (batch_idx + 1) % 10 == 0:
                pd.DataFrame(results).to_parquet(checkpoint_path, index=False)
                print(f"   💾 Checkpoint saved at batch {batch_idx + 1}.")

        except Exception as e:
            error_count += 1
            print(f"   ⚠️  Batch {batch_idx+1} error: {e}")
            # On error, append fallback records for this batch
            for profile in user_profiles:
                results.append({
                    "from_id": profile["from_id"],
                    "persona_codename": "CLASSIFICATION_ERROR",
                    "confidence": 0.0,
                    "justification": f"API error: {str(e)[:100]}"
                })

        time.sleep(1.0)  # Rate limit buffer

    # ── Final assembly ─────────────────────────────────────────────────────────
    results_df = pd.DataFrame(results)

    # Merge back with user features for the full enriched output
    final_df = user_features_df[["from_id", "from_username",
                                  "total_comments", "activity_span_days",
                                  "mean_hours_to_comment",
                                  "pct_comments_under_1h",
                                  "reply_ratio", "mean_likes_per_comment",
                                  "mean_word_count"]].merge(
        results_df, on="from_id", how="left"
    )

    final_df.to_parquet(output_path, index=False)
    print(f"\n✅ Stage 2 complete.")
    print(f"   Total classified: {len(final_df):,} users")
    print(f"   Errors: {error_count} batches")
    print(f"   Output: {output_path}")

    # ── Distribution summary ───────────────────────────────────────────────────
    dist = (final_df["persona_codename"]
            .value_counts(normalize=True)
            .mul(100).round(2))
    print("\n   📊 Persona Distribution:")
    for persona, pct in dist.items():
        print(f"      {persona:<35} {pct:.1f}%")

    return final_df



In [43]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — PIPELINE ORCHESTRATOR (Scheduled Runtime Entry Point)
# ─────────────────────────────────────────────────────────────────────────────
# This cell is the single entry point for the Colab Enterprise scheduler.
# Set PIPELINE_MODE at the top of the notebook before scheduling.
# ─────────────────────────────────────────────────────────────────────────────

def load_approved_taxonomy(path: str = TAXONOMY_JSON_PATH) -> list[dict]:
    """Load the human-approved MECE taxonomy from disk."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if data.get("status") != "APPROVED":
        raise ValueError(
            f"Taxonomy at '{path}' has not been approved. "
            "Complete human-in-the-loop review and set status='APPROVED'."
        )

    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty. Fill in the taxonomy before Stage 2.")

    print(f"✅ Approved taxonomy loaded: {len(taxonomy)} personas.")
    return taxonomy


def run_pipeline(mode: str = PIPELINE_MODE):
    """
    Main orchestrator.
    mode = "SAMPLE"  → Stage 0 + Stage 1
    mode = "FULL"    → Stage 0 + Stage 2 (requires approved taxonomy)
    mode = "ALL"     → Stage 0 + Stage 1 + Stage 2
    """
    print(f"\n{'#'*60}")
    print(f"  SHOW REEL PERSONA PIPELINE — MODE: {mode}")
    print(f"{'#'*60}\n")

    # Stage 0 always runs (feature engineering)
    # (Data already loaded in Cell 3, features built in Cell 4)
    # user_features and ig_media are available from prior cells.

    if mode in ("SAMPLE", "ALL"):
        candidates = run_stage1_taxonomy_discovery(
            user_features_df=user_features,
            n_sample=SAMPLE_N_USERS,
            batch_size=BATCH_SIZE_STAGE1,
            seed=SAMPLE_SEED,
        )
        save_taxonomy_for_review(candidates, TAXONOMY_JSON_PATH)

        if mode == "SAMPLE":
            print("\n⏸  Pipeline paused after Stage 1.")
            print("   ACTION: Review and approve taxonomy at:", TAXONOMY_JSON_PATH)
            return None

    if mode in ("FULL", "ALL"):
        taxonomy = load_approved_taxonomy(TAXONOMY_JSON_PATH)
        final_df = run_stage2_classification(
            user_features_df=user_features,
            taxonomy=taxonomy,
            ig_media_df=ig_media,
            batch_size=BATCH_SIZE_STAGE2,
            output_path=RESULTS_PATH,
            resume=True,  # Safe for scheduled restarts
        )
        return final_df

    return None


# ── EXECUTE ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    result = run_pipeline(mode=PIPELINE_MODE)


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



############################################################
  SHOW REEL PERSONA PIPELINE — MODE: SAMPLE
############################################################


STAGE 1 — Exploratory Taxonomy Discovery
  Sample: 500 users | Batch size: 20
  Model: gemini-2.5-flash



Stage 1 batches:   0%|          | 0/25 [00:00<?, ?it/s]

   ⚠️  Batch 1 error: Unterminated string starting at: line 7 column 7 (char 242)


Stage 1 batches:   4%|▍         | 1/25 [00:12<04:48, 12.02s/it]

...

   ⚠️  Batch 24 error: Unterminated string starting at: line 7 column 7 (char 267)


Stage 1 batches:  96%|█████████▌| 24/25 [04:40<00:11, 11.54s/it]

   ⚠️  Batch 25 error: Unterminated string starting at: line 7 column 7 (char 273)


Stage 1 batches: 100%|██████████| 25/25 [04:52<00:00, 11.68s/it]


✅ Stage 1 complete. Total raw candidates: 0
✅ Raw taxonomy candidates saved to: outputs/taxonomy.json
   ⚠️  HUMAN ACTION REQUIRED: Review, consolidate, and set status='APPROVED'.

⏸  Pipeline paused after Stage 1.
   ACTION: Review and approve taxonomy at: outputs/taxonomy.json


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — OPTIONAL: UPLOAD OUTPUTS TO GCS
# ─────────────────────────────────────────────────────────────────────────────

def upload_outputs_to_gcs(local_dir: str = "outputs/", gcs_path: str = GCS_BUCKET):
    """Upload all pipeline outputs to GCS for persistence across runtimes."""
    from google.cloud import storage

    client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = gcs_path.replace("gs://", "").split("/")[0]
    prefix      = "/".join(gcs_path.replace("gs://", "").split("/")[1:])
    bucket      = client.bucket(bucket_name)

    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        blob_path  = f"{prefix}/{fname}" if prefix else fname
        blob       = bucket.blob(blob_path)
        blob.upload_from_filename(local_path)
        print(f"   ☁️  Uploaded: {local_path} → gs://{bucket_name}/{blob_path}")

# Uncomment to upload after pipeline completes:
# upload_outputs_to_gcs()



In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — VALIDATION & QUICK QA
# ─────────────────────────────────────────────────────────────────────────────

def validate_classification_output(results_path: str = RESULTS_PATH):
    """
    Quick QA checks on the classification output:
    - Coverage rate (% users successfully classified)
    - Confidence distribution
    - MECE check (each user has exactly one persona)
    - Low-confidence flagging (< 0.4)
    """
    df = pd.read_parquet(results_path)

    total        = len(df)
    classified   = df["persona_codename"].notna() & \
                   (df["persona_codename"] != "CLASSIFICATION_ERROR")
    coverage_pct = classified.sum() / total * 100

    print(f"\n📋 CLASSIFICATION QA REPORT")
    print(f"   Total users        : {total:,}")
    print(f"   Successfully classified: {classified.sum():,} ({coverage_pct:.1f}%)")
    print(f"   Errors             : {(~classified).sum():,}")

    sub = df[classified].copy()
    sub["confidence"] = pd.to_numeric(sub["confidence"], errors="coerce")

    print(f"\n   Confidence Distribution:")
    print(f"     Mean   : {sub['confidence'].mean():.3f}")
    print(f"     Median : {sub['confidence'].median():.3f}")
    print(f"     < 0.40 : {(sub['confidence'] < 0.4).sum():,} users (low confidence, flag for review)")

    # MECE check: no user appears twice
    dupes = df["from_id"].duplicated().sum()
    print(f"\n   MECE Check: {dupes} duplicate user assignments "
          f"({'⚠️  FAIL' if dupes > 0 else '✅ PASS'})")

    return df

# Uncomment after Stage 2 completes:
# qa_df = validate_classification_output()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — COMMENT TEXT PROCESSING & JSONL EXPORT FOR LLM
# ─────────────────────────────────────────────────────────────────────────────
# Comprehensive preprocessing of comment text + export to JSONL format
# for direct feeding to LLM (inference or fine-tuning).
# Designed to run on Colab Enterprise with output to GCS.

import json
from typing import Optional, Dict, Any

def advanced_text_preprocessing(text: str) -> Dict[str, Any]:
    """
    Comprehensive comment text preprocessing:
    - Normalizes whitespace
    - Removes URLs
    - Preserves emoji but extracts count
    - Detects language
    - Flags mentions and hashtags
    - Returns both cleaned and original text

    Returns dict with:
      - text_original: unchanged raw text
      - text_cleaned: whitespace normalized + URLs removed
      - text_no_emoji: cleaned text with emojis removed (for strict NLP)
      - language: detected language code
      - emoji_count: number of emoji characters
      - mention_count: number of @mentions
      - hashtag_count: number of #hashtags
      - is_reply_text: True if text starts with @mention pattern
      - has_multiple_languages: True if mixed-script content detected
    """
    if not isinstance(text, str) or not text.strip():
        return {
            "text_original": "",
            "text_cleaned": "",
            "text_no_emoji": "",
            "language": None,
            "emoji_count": 0,
            "mention_count": 0,
            "hashtag_count": 0,
            "is_reply_text": False,
            "has_multiple_languages": False,
        }

    text_orig = text.strip()

    # Remove URLs (http/https/www patterns)
    text_no_urls = re.sub(r'https?://\S+|www\.\S+', '', text_orig)

    # Count and extract emoji
    emoji_count = emoji_lib.emoji_count(text_no_urls)

    # Remove emoji for strict text processing
    text_no_emoji_version = emoji_lib.replace_emoji(text_no_urls, "")

    # Normalize whitespace: multiple spaces → single space, strip newlines
    text_cleaned = re.sub(r'\s+', ' ', text_no_urls).strip()
    text_no_emoji = re.sub(r'\s+', ' ', text_no_emoji_version).strip()

    # Count mentions and hashtags
    mention_count = len(re.findall(r'@\w+', text_cleaned))
    hashtag_count = len(re.findall(r'#\w+', text_cleaned))

    # Check if text starts with mention (common in replies within IG)
    is_reply_text = bool(re.match(r'^\s*@\w+', text_orig))

    # Language detection
    try:
        detected_lang = langdetect.detect(text_no_emoji if text_no_emoji.strip() else "x")
    except:
        detected_lang = None

    # Heuristic: detect if text contains multiple script blocks (Latin + Cyrillic, etc.)
    has_multiple_scripts = bool(
        re.search(r'[a-zA-Z]', text_cleaned) and
        re.search(r'[^\x00-\x7F]', text_cleaned)  # non-ASCII blocks
    )

    return {
        "text_original": text_orig,
        "text_cleaned": text_cleaned,
        "text_no_emoji": text_no_emoji,
        "language": detected_lang,
        "emoji_count": emoji_count,
        "mention_count": mention_count,
        "hashtag_count": hashtag_count,
        "is_reply_text": is_reply_text,
        "has_multiple_languages": has_multiple_scripts,
    }


def export_comments_to_jsonl(
    comments_df: pd.DataFrame,
    output_gcs_path: str,
    include_features: bool = True,
    max_records: Optional[int] = None,
) -> int:
    """
    Export processed comments to JSONL format for LLM consumption.

    Each line is a JSON object with structure:
    {
        "id": comment_id,
        "user_id": from_id,
        "username": from_username,
        "text_original": raw text,
        "text_cleaned": preprocessed text,
        "text_no_emoji": cleaned without emoji,
        "metadata": {
            "language": language code,
            "emoji_count": int,
            "mention_count": int,
            "hashtag_count": int,
            "is_reply": bool,
            "timestamp": ISO timestamp,
            "likes": int,
            "media_id": str,
            ... [optional feature fields if include_features=True]
        }
    }

    Args:
        comments_df: DataFrame with comment data
        output_gcs_path: GCS destination (e.g., gs://bucket/path/comments.jsonl)
        include_features: If True, include quantitative features in metadata
        max_records: If set, sample this many records (for testing)

    Returns:
        Number of records written
    """
    from google.cloud import storage

    print(f"\n📝 Processing {len(comments_df):,} comments for JSONL export...")

    df = comments_df.copy()
    if max_records:
        df = df.sample(n=min(max_records, len(df)), random_state=42)

    # Apply advanced preprocessing
    print("   ⚙️  Running advanced text preprocessing...")
    text_features = df["text"].apply(advanced_text_preprocessing)

    # Create JSONL records
    records = []
    for idx, (_, row) in enumerate(df.iterrows()):
        text_proc = text_features.iloc[idx]

        metadata = {
            "language": text_proc["language"],
            "emoji_count": int(text_proc["emoji_count"]),
            "mention_count": int(text_proc["mention_count"]),
            "hashtag_count": int(text_proc["hashtag_count"]),
            "is_reply": bool(text_proc["is_reply_text"]),
            "timestamp": str(row.get("timestamp", "")),
            "likes": int(row.get("like_count", 0)),
            "media_id": str(row.get("media_id", "")),
        }

        # Optionally include quantitative features (from CELL 4)
        if include_features and all(col in df.columns for col in [
            "word_count", "char_length", "has_emoji", "has_question",
            "has_exclaim", "is_reply"
        ]):
            metadata.update({
                "word_count": int(row.get("word_count", 0)),
                "char_length": int(row.get("char_length", 0)),
                "has_question": bool(row.get("has_question", 0)),
                "has_exclaim": bool(row.get("has_exclaim", 0)),
            })

        record = {
            "id": str(row.get("comment_id", f"comment_{idx}")),
            "user_id": str(row.get("from_id", "")),
            "username": str(row.get("from_username", "")),
            "text_original": text_proc["text_original"],
            "text_cleaned": text_proc["text_cleaned"],
            "text_no_emoji": text_proc["text_no_emoji"],
            "metadata": metadata,
        }
        records.append(record)

    # Write to JSONL in memory
    jsonl_buffer = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)

    # Upload to GCS
    print(f"   ☁️  Uploading to {output_gcs_path}...")
    try:
        client = storage.Client()
        bucket_name = output_gcs_path.replace("gs://", "").split("/")[0]
        blob_path = "/".join(output_gcs_path.replace("gs://", "").split("/")[1:])

        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_path)
        blob.upload_from_string(jsonl_buffer, content_type="application/x-ndjson")

        print(f"   ✅ Exported {len(records):,} records to: {output_gcs_path}")
        return len(records)
    except Exception as e:
        print(f"   ⚠️  GCS upload failed: {e}")
        print(f"   💾 Falling back to local export...")
        local_path = output_gcs_path.replace("gs://", "").replace("/", "_")
        with open(local_path, "w", encoding="utf-8") as f:
            f.write(jsonl_buffer)
        print(f"   ✅ Exported {len(records):,} records to: {local_path}")
        return len(records)


# ── Example: Export all comments to JSONL ────────────────────────────────────
# Uncomment and adjust path as needed:
# export_comments_to_jsonl(
#     comments_df=ig_comments,
#     output_gcs_path=f"gs://{GCS_BUCKET}/comments_for_llm.jsonl",
#     include_features=True,
#     max_records=None  # Set to 10000 to test with subset first
# )

print("✅ Comment text processing + JSONL export functions ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — EXAMPLE: PROCESS & EXPORT SAMPLE COMMENTS TO JSONL
# ─────────────────────────────────────────────────────────────────────────────
# Demonstrates the comment preprocessing pipeline with a small sample.
# Set max_records to test before running on the full dataset.

# Test with a small sample first (e.g., 100 comments)
print("🧪 Testing comment preprocessing on a sample...\n")

# Apply preprocessing to a small sample
sample_size = 100
sample_comments = ig_comments.head(sample_size)
sample_preprocessed = sample_comments["text"].apply(advanced_text_preprocessing)

# Display stats
print(f"Sample Stats ({sample_size} comments):")
print(f"  Avg emoji count:    {sample_preprocessed.apply(lambda x: x['emoji_count']).mean():.2f}")
print(f"  Avg mention count:  {sample_preprocessed.apply(lambda x: x['mention_count']).mean():.2f}")
print(f"  Avg hashtag count:  {sample_preprocessed.apply(lambda x: x['hashtag_count']).mean():.2f}")
print(f"  Language diversity: {len(sample_preprocessed.apply(lambda x: x['language']).unique())} unique languages")
print(f"  Mixed-script text:  {sample_preprocessed.apply(lambda x: x['has_multiple_languages']).sum()} comments")

# Show 3 examples of processed comments
print(f"\n📋 Sample processed comments:\n")
for i in range(min(3, len(sample_preprocessed))):
    proc = sample_preprocessed.iloc[i]
    print(f"  Example {i+1}:")
    print(f"    Original:  {proc['text_original'][:80]}")
    print(f"    Cleaned:   {proc['text_cleaned'][:80]}")
    print(f"    Language:  {proc['language']} | Emoji: {proc['emoji_count']} | Mentions: {proc['mention_count']}")
    print()

# ── Full export (uncomment to run on entire dataset) ──────────────────────────
# To export ALL comments (may take a few minutes):
#
# export_comments_to_jsonl(
#     comments_df=ig_comments,
#     output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm.jsonl",
#     include_features=True,
#     max_records=None
# )

# ── Or test with a smaller subset for validation ──────────────────────────────
# To test with 5000 comments first:
#
# export_comments_to_jsonl(
#     comments_df=ig_comments,
#     output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm_sample_5k.jsonl",
#     include_features=True,
#     max_records=5000
# )

print("\n✅ Preprocessing pipeline tested. Ready to export full dataset when needed.")